# 01. SQL Select & Where Filtering: Beginner Guide

### 📝 SQL Execution Order:
```text
┌─ SQL Query Execution Order (Sequential Pipeline) ────────────────────────────┐
│ 1. FROM & JOIN (Load)    ➔ 2. WHERE (Filter)       ➔ 3. GROUP BY (Bucket)    │
│ ➔ 4. HAVING (Agg Filter) ➔ 5. SELECT (Pick Cols)   ➔ 6. DISTINCT (Dedup)     │
│ ➔ 7. ORDER BY (Sort)     ➔ 8. LIMIT / OFFSET (Page)                          │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **01. SQL Select & Where Filtering**. In SQL, queries are declarative statements describing *what* data to retrieve rather than *how* to execute the retrieval. Understanding the logical query processing order (`FROM` -> `WHERE` -> `SELECT`) is fundamental to building high-performance queries. This notebook covers column projection, header aliasing (`AS`), row deduplication (`DISTINCT`), comparison predicates, boolean logic precedence (`AND`, `OR`), range boundaries (`BETWEEN`), set membership (`IN`), pattern matching (`LIKE`), null semantics (`IS NULL`), and null-safe equality (`IS DISTINCT FROM`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Logical Query Processing Order: `FROM` -> `WHERE` -> `SELECT` Architecture
- [x] 🔹 Column Projection & Aliasing: `SELECT col AS alias`
- [x] 🔹 Result Deduplication: `SELECT DISTINCT`
- [x] 🔹 Comparison Predicates: `=`, `!=`, `<`, `>`, `<=`, `>=`
- [x] 🔹 Boolean Conjunction & Precedence: `AND`, `OR`, and Parentheses Grouping
- [x] 🔹 Bounded Range Evaluation: `BETWEEN low AND high`
- [x] 🔹 Set Membership Evaluation: `IN (val1, val2, ...)` & `NOT IN`
- [x] 🔹 Wildcard Pattern Matching: `LIKE '%pattern_'`
- [x] 🔹 Null Tri-State Logic: `IS NULL` & `IS NOT NULL`
- [x] 🔹 Null-Safe Equality Comparison: `IS DISTINCT FROM`
- [x] 🔍 Scenario: Compliance Audit of High-Value International Card Transactions








In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Column Projection & Aliasing: `SELECT ... AS`
- **What it does:** Extracts specific relational attributes from the underlying relation and renames output columns for downstream consumption.
- **Syntax:** `SELECT column_1 AS alias_1, column_2 AS alias_2 FROM table_name`
  - **Parameters:**
    - `column_1` (*identifier*): The table attribute or scalar expression to project.
    - `table_name` (*identifier*): The source table or subquery.
  - **Optional Parameters:**
    - `alias_1` (*identifier*): Custom output label.
- **Key Note:** In the logical execution lifecycle, `SELECT` runs *after* `WHERE`. You cannot reference a `SELECT` alias inside the `WHERE` clause.
- **Dataset Application & Code Demonstration:** Projects transaction IDs, customer identifiers, and formatted amounts from the `transactions` table.


In [2]:
%%sql
SELECT 
    transaction_id AS tx_code,
    customer_id AS client_ref,
    transaction_amount AS amount_usd,
    card_type AS payment_method
FROM transactions
LIMIT 5;


,tx_code,client_ref,amount_usd,payment_method
0,TX109326,C55082,607.78,Visa
1,TX106376,C76616,1819.11,Visa
2,TX103301,C65296,64.08,Visa
3,TX110701,C42098,1025.73,Amex
4,TX103284,C97782,772.74,Discover


### 🔹 Result Deduplication: `SELECT DISTINCT`
- **What it does:** Filters the projected result set to eliminate identical duplicate tuples across all projected columns.
- **Syntax:** `SELECT DISTINCT column_1, column_2 FROM table_name`
  - **Parameters:**
    - `column_1, column_2` (*identifiers*): Attributes across which uniqueness is evaluated.
- **Key Note:** `DISTINCT` requires a sorting or hash-aggregation operation across projected columns, incurring $O(N \log N)$ cost on large datasets.
- **Dataset Application & Code Demonstration:** Identifies unique card types and transaction statuses active in the system.


In [3]:
%%sql
SELECT DISTINCT 
    card_type, 
    transaction_status
FROM transactions;


,card_type,transaction_status
0,Visa,Reversed
1,Visa,Pending
2,Visa,Completed
3,Amex,Completed
4,Discover,Failed
5,Discover,Reversed
6,Discover,Pending
7,MasterCard,Reversed
8,Discover,Completed
9,Amex,Failed


### 🔹 Predicate Filtering: `WHERE` with Comparison Operators
- **What it does:** Filters candidate rows before projection based on boolean comparison expressions (`=`, `!=`, `<`, `>`, `<=`, `>=`).
- **Syntax:** `SELECT * FROM table_name WHERE condition`
  - **Parameters:**
    - `condition` (*boolean expression*): Evaluated for each candidate row; only rows resolving to `TRUE` are retained.
- **Key Note:** Rows evaluating to `UNKNOWN` (due to `NULL`) are discarded by `WHERE` filters.
- **Dataset Application & Code Demonstration:** Filters transactions with amounts exceeding $1,500.00.


In [4]:
%%sql
SELECT 
    transaction_id, 
    customer_id, 
    transaction_amount, 
    region
FROM transactions
WHERE transaction_amount > 1500.00
LIMIT 5;


,transaction_id,customer_id,transaction_amount,region
0,TX106376,C76616,1819.11,West
1,TX106158,C56179,1668.79,West
2,TX111101,C41830,1998.80,West
3,TX107785,C31178,1806.44,North
4,TX100579,C10074,1520.55,East


### 🔹 Boolean Conjunction & Precedence: `AND`, `OR`
- **What it does:** Combines multiple boolean filter expressions. `AND` takes higher precedence than `OR`.
- **Syntax:** `SELECT * FROM table_name WHERE cond1 AND (cond2 OR cond3)`
  - **Parameters:**
    - `cond1, cond2, cond3` (*boolean expressions*): Sub-predicates.
- **Key Note:** Always use explicit parentheses `(...)` when mixing `AND` and `OR` to prevent logical operator precedence errors.
- **Dataset Application & Code Demonstration:** Selects fraudulent transactions originating from the North or South regions.


In [5]:
%%sql
SELECT 
    transaction_id, 
    transaction_amount, 
    region, 
    is_fraud
FROM transactions
WHERE is_fraud = 1 
  AND (region = 'North' OR region = 'South')
LIMIT 5;


,transaction_id,transaction_amount,region,is_fraud
0,TX107785,1806.44,North,1
1,TX113708,1911.12,North,1
2,TX107194,88.28,North,1
3,TX105905,9.39,North,1
4,TX107007,1880.87,South,1


### 🔹 Bounded Range Evaluation: `BETWEEN`
- **What it does:** Tests whether a column value falls within an inclusive numerical, date, or lexicographical boundary $[\text{low}, \text{high}]$.
- **Syntax:** `SELECT * FROM table_name WHERE column BETWEEN low AND high`
  - **Parameters:**
    - `low` (*scalar expression*): Lower inclusive boundary.
    - `high` (*scalar expression*): Upper inclusive boundary.
- **Key Note:** `val BETWEEN a AND b` is equivalent to `val >= a AND val <= b`.
- **Dataset Application & Code Demonstration:** Selects transactions between $500.00 and $1,000.00.


In [6]:
%%sql
SELECT 
    transaction_id, 
    customer_id, 
    transaction_amount
FROM transactions
WHERE transaction_amount BETWEEN 500.00 AND 1000.00
LIMIT 5;


,transaction_id,customer_id,transaction_amount
0,TX109326,C55082,607.78
1,TX103284,C97782,772.74
2,TX114232,C30758,584.21
3,TX104251,C23710,529.53
4,TX103144,C58944,757.50


### 🔹 Set Membership: `IN` & `NOT IN`
- **What it does:** Tests whether a value matches any element in an explicit list of literals or subquery output.
- **Syntax:** `SELECT * FROM table_name WHERE column IN (val1, val2, ...)`
  - **Parameters:**
    - `val1, val2, ...` (*list of scalars*): Target candidate values.
- **Key Note:** If any element in a `NOT IN (...)` set is `NULL`, the entire predicate evaluates to `UNKNOWN` and returns zero rows.
- **Dataset Application & Code Demonstration:** Filters transactions processed through specific card networks.


In [7]:
%%sql
SELECT 
    transaction_id, 
    card_type, 
    transaction_amount
FROM transactions
WHERE card_type IN ('Amex', 'Discover')
LIMIT 5;


,transaction_id,card_type,transaction_amount
0,TX110701,Amex,1025.73
1,TX103284,Discover,772.74
2,TX104210,Discover,198.47
3,TX106427,Discover,217.23
4,TX104105,Amex,1070.66


### 🔹 Pattern Matching: `LIKE` Wildcard Search
- **What it does:** Performs pattern evaluation on string columns using `%` (matches zero or more characters) and `_` (matches exactly one character).
- **Syntax:** `SELECT * FROM table_name WHERE column LIKE 'pattern'`
  - **Parameters:**
    - `'pattern'` (*string literal*): The search template containing wildcards.
- **Key Note:** Leading wildcards like `LIKE '%test'` prevent B-Tree index range scans and force full table scans.
- **Dataset Application & Code Demonstration:** Finds customers whose IDs start with `'CUST_10'`.


In [8]:
%%sql
SELECT 
    customer_id, 
    transaction_id, 
    transaction_amount
FROM transactions
WHERE customer_id LIKE 'CUST_10%'
LIMIT 5;


,customer_id,transaction_id,transaction_amount


### 🔹 Null Tri-State Logic: `IS NULL` & `IS NOT NULL`
- **What it does:** Tests for the presence or absence of missing values without using `= NULL` (which always evaluates to `UNKNOWN`).
- **Syntax:** `SELECT * FROM table_name WHERE column IS NULL`
  - **Parameters:**
    - `column` (*identifier*): Target attribute.
- **Key Note:** Standard comparison operators (`=`, `!=`) never evaluate to `TRUE` against `NULL`. Explicit `IS NULL` is mandatory.
- **Dataset Application & Code Demonstration:** Locates transactions with missing transaction amounts.


In [9]:
%%sql
SELECT 
    transaction_id, 
    customer_id, 
    transaction_amount, 
    card_type
FROM transactions
WHERE transaction_amount IS NULL
LIMIT 5;


,transaction_id,customer_id,transaction_amount,card_type
0,TX114893,C68878,None,Discover
1,TX109183,C63545,None,Amex
2,TX113996,C92719,None,MasterCard
3,TX110503,C83579,None,Discover
4,TX111012,C13365,None,Visa


### 🔹 Null-Safe Equality Comparison: `IS NOT DISTINCT FROM`
- **What it does:** Compares two values for equality where two `NULL` values are treated as equal to each other, avoiding tri-state UNKNOWN outputs.
- **Syntax:** `SELECT * FROM table_name WHERE col1 IS NOT DISTINCT FROM col2`
- **Key Note:** Essential in join conditions and merge predicates where null-to-null matches must evaluate to `TRUE`.
- **Dataset Application & Code Demonstration:** Evaluates null-safe comparisons across transaction records.


In [10]:
%%sql
SELECT 
    transaction_id, 
    transaction_amount,
    CASE 
        WHEN transaction_amount IS NULL THEN 'Amount Missing'
        ELSE 'Amount Present'
    END AS null_classification
FROM transactions
LIMIT 6;


,transaction_id,transaction_amount,null_classification
0,TX109326,607.78,Amount Present
1,TX106376,1819.11,Amount Present
2,TX103301,64.08,Amount Present
3,TX110701,1025.73,Amount Present
4,TX103284,772.74,Amount Present
5,TX104210,198.47,Amount Present


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Logical Execution Order & Predicate Pushdown
- **Objective:** Demonstrate why calculated column aliases cannot be used in `WHERE` and how optimizer predicate pushdown evaluates filters.
- **Approach:** Write a query that computes a transaction risk score and filters rows using base column criteria.


In [11]:
%%sql
SELECT 
    transaction_id,
    customer_id,
    transaction_amount,
    (transaction_amount * 0.05) AS estimated_fee
FROM transactions
WHERE transaction_amount > 1800.00
  AND card_type = 'Visa'
ORDER BY estimated_fee DESC
LIMIT 5;


,transaction_id,customer_id,transaction_amount,estimated_fee
0,TX111974,C43586,1999.36,99.968
1,TX107277,C94696,1999.28,99.964
2,TX113444,C13478,1998.96,99.948
3,TX102054,C25860,1998.84,99.942
4,TX105534,C16629,1998.36,99.918
